# Aufgabe 10.3 – AI4I mit Seaborn visualisieren

**Status: AWAITING HUMAN REVIEW.** Implementierung, Berechnungen und technische
Prüfung: Codex. Keine studentische Freigabe dokumentiert.

[Offizielle Aufgabe](https://noxthot.github.io/ws26_industrial_datascience_manuscript/1_act1/6_self_study_session1.html),
vollständig gelesen am 26.10.2026. Wir untersuchen alle 10.000 synthetischen
Beobachtungen aus [UCI AI4I 2020](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset),
DOI 10.24432/C5HS5C, CC BY 4.0. Quelle/Prüfsumme: `data/raw/README.md`.

Vorbereitung: `uv sync --locked`, `uv run python scripts/download_ai4i.py`.
Reproduktion: `uv run python scripts/build_exercise_10_3.py`. Das Notebook
verwendet danach nur lokale Daten. Die Aufgabe verlangt keinen PDF-Export;
HTML und alle PNG-Figuren werden gespeichert. Keine 10.4-Quarto-Arbeit.

## Daten und Analyseentscheidungen

Der geprüfte Loader aus 10.2 kontrolliert unveränderte Rohbytes, Spalten,
Zeilenzahl und Kennungen. `numeric` enthält ausschließlich die drei Messgrößen;
UDI bleibt separat für die ausdrücklich verlangte Linie. Keine Imputation,
Normalisierung, Filterung oder Stichprobe. UDI ist **kein Zeitstempel**.
Ein Seed reproduziert nur den Jitter, nicht eine vermeintliche Messunsicherheit.

In [ ]:
from pathlib import Path
import os
import sys
import json

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root)) if str(root) not in sys.path else None
os.environ.setdefault("MPLCONFIGDIR", str(root / ".tools/matplotlib"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Image, Markdown
from sessions.ch10.ai4i_data import load_ai4i, verify_file
from sessions.ch10.ai4i_eda import (
    MEASUREMENTS, TORQUE, SPEED, AIR, SEED, add_bins,
    iqr_summary, strongest_pair,
)

data = load_ai4i()
numeric = data.loc[:, MEASUREMENTS].copy()
assert numeric.shape == (10000, 3)
out = root / "outputs/ch10/exercise_10_3"
out.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

def save_show(fig, name):
    path = out / f"{name}.png"
    fig.savefig(path, dpi=140, bbox_inches="tight")
    display(Image(filename=str(path)))
    plt.close(fig)

print(f"Daten: {len(data)} Zeilen, drei Messgrößen; Seaborn {sns.__version__}")
print(f"SHA256: {verify_file()}")
display(numeric.describe())

## 1. Drehmoment über UDI

Die Linie verbindet alle Beobachtungen in Originalreihenfolge. `estimator=None`
verhindert Aggregation; es wird kein Unsicherheitsband dargestellt. Dichte
Schwankungen über eine Zeilenkennung dürfen nicht als zeitlicher Prozess
interpretiert werden.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
sns.lineplot(data=data, x="UDI", y=TORQUE, estimator=None,
             errorbar=None, sort=False, linewidth=0.5, ax=ax)
ax.set(title="AI4I: Drehmoment über UDI", xlabel="UDI (Zeilenkennung)",
       ylabel=TORQUE)
save_show(fig, "01_line")

## 2. Verteilungen – Histogramme

Je Messgröße werden 30 Bins verwendet. Anzahl und Grenzen der Bins beeinflussen
den visuellen Eindruck. Die x-Achsen behalten ihre Einheiten und eigenen Skalen;
die y-Achsen zählen Beobachtungen, nicht Wahrscheinlichkeitsdichten.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for ax, column in zip(axes, MEASUREMENTS):
    sns.histplot(data=numeric, x=column, bins=30, ax=ax, color="steelblue")
    ax.set(title=column, xlabel=column, ylabel="Anzahl")
fig.suptitle("Histogramme aller drei Messgrößen")
save_show(fig, "02_histograms")

## 3. Boxplots und auffällige Werte

Boxplots verdichten Median, Quartile und Streuung. Punkte außerhalb der
1,5-IQR-Zäune sind hier **heuristisch auffällig**, nicht automatisch fehlerhaft.
Die jeweilige Messgröße erhält eine eigene Achse mit Originaleinheit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for ax, column in zip(axes, MEASUREMENTS):
    sns.boxplot(data=numeric, y=column, whis=1.5, fliersize=2, ax=ax,
                color="lightsteelblue")
    ax.set(title=column, ylabel=column, xlabel="Alle Beobachtungen")
fig.suptitle("Boxplots: Quartile und 1,5-IQR-Markierungen")
save_show(fig, "03_boxplots")
iqr = iqr_summary(numeric)
iqr.to_csv(out / "iqr_summary.csv")
display(iqr)

## 4. Violinplots im Vergleich zu Boxplots

Die Violinbreite ist eine Kernel-Dichteschätzung (KDE), keine Anzahl einzelner
Beobachtungen. Bandbreite und Stichprobe beeinflussen die Form. `cut=0` begrenzt
die Darstellung auf beobachtete Werte; innere Quartile ergänzen die Dichte.
Gegenüber Boxplots wird Verteilungsform sichtbar, aber auch geglättet und damit
von einer zusätzlichen Modellierungsentscheidung abhängig.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for ax, column in zip(axes, MEASUREMENTS):
    sns.violinplot(data=numeric, y=column, cut=0, bw_adjust=1,
                   inner="quart", color="lightsteelblue", ax=ax)
    ax.set(title=column, ylabel=column, xlabel="KDE und Quartile")
fig.suptitle("Violinplots: geglättete Verteilungsform")
save_show(fig, "04_violins")

## 5. Torque gegen Rotational speed – Temperatur als Farbe

`sns.scatterplot` bildet Drehzahl auf x, Drehmoment auf y und Lufttemperatur auf
die kontinuierliche Farbskala ab. Transparenz begrenzt die Überlagerung aller
10.000 Punkte. Die Farblegende zeigt Kelvin; Farbunterschiede beweisen keinen
Einfluss der Temperatur.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
sns.scatterplot(data=data, x=SPEED, y=TORQUE, hue=AIR, palette="viridis",
                s=9, alpha=0.45, linewidth=0, legend="brief", ax=ax)
ax.set(title="Drehmoment und Drehzahl, farbig nach Lufttemperatur")
ax.legend(title=AIR)
save_show(fig, "05_scatter")

## 6. Paarweise Beziehungen

`sns.pairplot` zeigt alle drei Messgrößen, ihre paarweisen Punktwolken und
Histogramme auf der Diagonalen. Unterschiedliche Achsenskalierungen sind nötig,
weil Einheiten und Wertebereiche verschieden sind. Alle Beobachtungen werden
verwendet; dichte Bereiche können sich weiterhin überdecken.

In [ ]:
grid = sns.pairplot(data=numeric, vars=MEASUREMENTS, diag_kind="hist",
                    height=2.8, plot_kws={"s": 7, "alpha": 0.2, "linewidth": 0},
                    diag_kws={"bins": 30})
grid.figure.suptitle("AI4I: alle drei Messgrößen paarweise", y=1.02)
save_show(grid.figure, "06_pairplot")

## 7. Korrelation

Pearson misst den linearen Zusammenhang. Die Heatmap verwendet die feste Skala
−1 bis +1 und annotiert alle Koeffizienten. Das stärkste Paar wird nach dem
Betrag der Off-Diagonale gewählt. Spearman ergänzt die Betrachtung monotoner,
möglicherweise nichtlinearer Muster. Keine Korrelation belegt Kausalität.

In [ ]:
pearson = numeric.corr(method="pearson")
spearman = numeric.corr(method="spearman")
pearson.to_csv(out / "pearson.csv")
spearman.to_csv(out / "spearman.csv")
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
sns.heatmap(pearson, annot=True, fmt=".3f", vmin=-1, vmax=1,
            center=0, cmap="vlag", square=True, ax=ax,
            cbar_kws={"label": "Pearson r"})
ax.set_title("Pearson-Korrelation der drei Messgrößen")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
save_show(fig, "07_heatmap")
display(spearman)
a, b, r = strongest_pair(pearson)
display(Markdown(f"Stärkstes Pearson-Paar: **{a} / {b}**, r = **{r:.4f}**. "
                 f"Spearman dafür: **{spearman.loc[a, b]:.4f}**."))

## 8. Jointplot mit Regressionsgerade

Torque wird gegen Air temperature geplottet, inklusive Randverteilungen.
Die Gerade ist eine deskriptive lineare Zusammenfassung, kein Kausalmodell.
Ein Konfidenzband wird weggelassen (`ci=None`), da hier keine ungeprüfte
Unabhängigkeitsannahme in eine inferenzstatistische Aussage übersetzt werden soll.

In [ ]:
grid = sns.jointplot(data=numeric, x=AIR, y=TORQUE, kind="reg", height=7,
                     joint_kws={"ci": None,
                                 "scatter_kws": {"s": 5, "alpha": 0.15},
                                 "line_kws": {"color": "crimson"}})
grid.figure.suptitle("Drehmoment gegen Lufttemperatur: deskriptive Regression", y=1.03)
save_show(grid.figure, "08_joint_regression")

## 9. Stripplot nach Drehzahl-Bins

`pd.cut` teilt den beobachteten Drehzahlbereich in drei Intervalle gleicher
Breite, benannt low/medium/high. Das sind **keine Quantile**; Gruppengrößen dürfen
sich stark unterscheiden. Die exakten Grenzen und Häufigkeiten werden ausgegeben.
Alle Drehmomentwerte erscheinen als Punkte, Farbe und x-Position kennzeichnen
denselben Bin. Jitter (Seed 42) dient ausschließlich der Darstellung.

In [ ]:
binned, edges = add_bins(data)
(out / "bin_edges.json").write_text(json.dumps(edges, indent=2), encoding="utf-8")
speed_counts = binned["speed_band"].value_counts(sort=False)
speed_counts.to_csv(out / "speed_bin_counts.csv", header=["count"])
display(pd.DataFrame({"lower": edges["speed_edges"][:-1],
                      "upper": edges["speed_edges"][1:],
                      "count": speed_counts.to_numpy()}, index=speed_counts.index))
np.random.seed(SEED)
fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
sns.stripplot(data=binned, x="speed_band", y=TORQUE, hue="speed_band",
              order=["low", "medium", "high"], hue_order=["low", "medium", "high"],
              palette="Set2", jitter=0.25, alpha=0.25, size=2, legend="full", ax=ax)
ax.set(title="Drehmoment nach Drehzahl-Intervallen", xlabel="Drehzahl-Bin (rpm)", ylabel=TORQUE)
ax.legend(title="Drehzahl-Bin")
save_show(fig, "09_strip")

## 10. Countplot der Torque-Intervalle

Die zehn Intervalle gleicher Breite decken alle Drehmomentwerte ab; die
Intervallgrenzen sind zusätzlich als JSON gespeichert. `pd.cut` erweitert bei
ganzzahliger Bin-Anzahl den Rand leicht, damit die Extremwerte eingeschlossen
sind. Der Countplot zählt alle 10.000 Zeilen ohne Filterung.

In [ ]:
torque_counts = binned["torque_band"].value_counts(sort=False)
assert int(torque_counts.sum()) == len(data)
torque_counts.to_csv(out / "torque_bin_counts.csv", header=["count"])
fig, ax = plt.subplots(figsize=(12, 5), layout="constrained")
sns.countplot(data=binned, x="torque_band", order=binned["torque_band"].cat.categories,
              color="steelblue", ax=ax)
ax.set(title="Beobachtungszahlen in zehn Drehmoment-Intervallen",
       xlabel="Drehmoment-Intervall [Nm]", ylabel="Anzahl")
ax.tick_params(axis="x", labelrotation=35)
save_show(fig, "10_count")
display(torque_counts.to_frame("count"))

## Beobachtungen – tatsächlich berechnete Kennzahlen

Die folgende Tabelle ergänzt die visuelle Beurteilung mit reproduzierbaren
Kennzahlen. IQR-Flags werden nicht entfernt. Skewness beschreibt Asymmetrie,
ist aber weder Normalitätstest noch Beleg einer bestimmten Verteilung.

In [ ]:
display(iqr[["flagged_count", "skewness", "minimum", "maximum"]])
display(Markdown(
    f"**Auffällige Werte:** Torque {int(iqr.loc[TORQUE, 'flagged_count'])}, "
    f"Rotational speed {int(iqr.loc[SPEED, 'flagged_count'])}, "
    f"Air temperature {int(iqr.loc[AIR, 'flagged_count'])} außerhalb der 1,5-IQR-Zäune. "
    f"**Stärkste Beziehung:** {a} / {b}, Pearson r = {r:.4f}. "
    f"Drehmoment-Skewness {iqr.loc[TORQUE, 'skewness']:.3f}, "
    f"Drehzahl-Skewness {iqr.loc[SPEED, 'skewness']:.3f}, "
    f"Temperatur-Skewness {iqr.loc[AIR, 'skewness']:.3f}."
))

## Einordnung und Grenzen

Die Ausgaben müssen visuell zusammen mit den Kennzahlen beurteilt werden:
Histogramm und Violinform hängen von Bins bzw. KDE-Bandbreite ab; Boxplot-Flags
sind keine nachgewiesenen Fehler. Eine gekrümmte Punktwolke kann von einem
linearen Korrelationskoeffizienten unvollständig zusammengefasst werden.
Ein Vergleich mit Spearman beschreibt Monotonie, liefert aber keinen
Mechanismus. Die UCI-Daten sind synthetisch; UDI enthält keine Zeitangabe.
Farbliche Gruppierung und Regression beweisen keinen Einfluss oder Ursache.

Die endgültige datenbezogene Interpretation wird nach tatsächlicher Ausführung
und Sichtprüfung ergänzt. Studentische Prüfung und Freigabe stehen aus.

## Fragen für Human Review

1. Sind alle verlangten Diagramme und alle drei Messgrößen vorhanden und lesbar?
2. Wie unterscheiden sich Boxplot und Violinplot, und warum sind IQR-Flags keine bewiesenen Fehler?
3. Was zeigen Scatter, Pairplot und Pearson/Spearman über die stärkste Beziehung und mögliche Nichtlinearität?
4. Wie beeinflussen gleiche Bin-Breiten, ungleiche Gruppengrößen, Jitter und KDE die Darstellung?
5. Welche Schlussfolgerungen sind wegen synthetischer Daten und fehlender Zeitinformation nicht zulässig?

**AWAITING HUMAN REVIEW. Keine Freigabe, kein Merge, keine automatische Abgabe.**